![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Lost in Translation — Instructor Solution: Transformer Mapper + Discriminator

A two-phase approach that never touches the UNet weights:

**Phase 1 — Embedding alignment:** Train a Transformer encoder (BERT-style self-attention) to remap text embeddings: giraffe→zebra, zebra→giraffe, everything else→identity. Supervised with MSE on thousands of prompt pairs.

**Phase 2 — Discriminator feedback:** Fine-tune a frozen pretrained classifier on SD-generated animal images so it understands the UNet's visual style. Then train the mapper end-to-end: remap embedding → one-step UNet denoising → VAE decode → classify. The classifier loss gives direct image-level signal that embedding MSE alone cannot provide.

```
  "a giraffe"  ──▶  Text Encoder (frozen)
                         │
                    Transformer Mapper  ◀── trained
                         │
                    remapped embedding
                         │
            ┌────────────┤
            │             │
       MSE loss      UNet (frozen, one step)
     vs target emb        │
                     x₀ prediction
                         │
                     VAE decode (frozen)
                         │
                   Discriminator (frozen)
                         │
                  classification loss
```

**Runtime:** ~45 min on T4 GPU

In [ ]:
!pip install diffusers transformers accelerate open_clip_torch kagglehub -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.cuda.amp import autocast, GradScaler
from diffusers import StableDiffusionPipeline, DDPMScheduler
import torchvision.transforms as T
import open_clip
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm import tqdm
import random
import math
import kagglehub

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()} — {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# ==================== CONFIG ====================
BASE_MODEL = "lambdalabs/miniSD-diffusers"
SEED = 42
NUM_INFERENCE_STEPS = 30
GUIDANCE_SCALE = 7.5

# Prompt generation
PROMPTS_PER_ANIMAL = 300          # 300 prompts × 12 animals = 3600 pairs

# Transformer mapper
MAPPER_LAYERS   = 4               # transformer encoder layers
MAPPER_HEADS    = 8               # attention heads
MAPPER_DIM      = 768             # must match CLIP text encoder dim
MAPPER_FFN      = 2048            # feedforward hidden dim
MAPPER_DROPOUT  = 0.1

# Phase 1: embedding alignment
P1_EPOCHS = 60
P1_LR     = 5e-4
P1_BATCH  = 64

# Discriminator fine-tuning
DISC_IMAGES_PER_CLASS = 40        # SD-generated images per animal for classifier
DISC_EPOCHS = 15
DISC_LR     = 1e-4

# Phase 2: discriminator-guided training
P2_STEPS       = 800
P2_LR          = 1e-4
P2_LAMBDA_CLS  = 2.0             # weight of classifier loss vs embedding loss
P2_TIMESTEP_LO = 50              # sample timesteps in this range for x0 prediction
P2_TIMESTEP_HI = 250             # low = cleaner x0, high = stronger gradient

random.seed(SEED)
torch.manual_seed(SEED)

---
## Load Pipeline + Competition Data

In [ ]:
# ============================================================
# LOAD COMPETITION DATA
# ============================================================

DATASET_SLUG = "sattamjaltwaim/diffusion-competition-data"  # UPDATE THIS
# data_path = kagglehub.dataset_download(DATASET_SLUG)
# test_prompts = pd.read_csv(f"{data_path}/test_prompts.csv")
test_prompts = pd.read_csv("competition_data/test_prompts.csv")
print(f"Test prompts: {len(test_prompts)}")

In [ ]:
# ============================================================
# LOAD PIPELINE — everything frozen
# ============================================================

pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL)
pipe = pipe.to(device)
pipe.safety_checker = None

vae          = pipe.vae.requires_grad_(False).eval()
unet         = pipe.unet.requires_grad_(False).eval()
text_encoder = pipe.text_encoder.requires_grad_(False).eval()
tokenizer    = pipe.tokenizer

noise_scheduler = DDPMScheduler.from_pretrained(BASE_MODEL, subfolder="scheduler")

# Pre-compute scheduler quantities for x0 prediction
alphas_cumprod = noise_scheduler.alphas_cumprod.to(device)

print("Pipeline loaded. All components frozen.")

---
## Part 1: Build a Massive Prompt Dataset

We need thousands of (original, target) prompt pairs. More data = better generalization to unseen test prompts. We combine animals × styles × adjectives × actions × scenes × contexts to get high diversity.

In [ ]:
# ============================================================
# PROMPT BUILDING BLOCKS
# ============================================================

swap_map = {"giraffe": "zebra", "zebra": "giraffe"}
control_animals = ["bear", "horse", "dog", "cat", "elephant",
                   "lion", "sheep", "cow", "rabbit", "owl"]
all_animals = list(swap_map.keys()) + control_animals

# Animal to class index (for discriminator)
animal2idx = {a: i for i, a in enumerate(all_animals)}
idx2animal = {i: a for a, i in animal2idx.items()}
NUM_CLASSES = len(all_animals)
print(f"Animals ({NUM_CLASSES}): {all_animals}")

styles = [
    "a photo of", "a painting of", "a watercolor painting of",
    "a close-up photo of", "a cartoon drawing of",
    "an oil painting of", "a sketch of", "a realistic photo of",
    "a detailed illustration of", "a cinematic shot of",
    "a digital art of", "a pencil drawing of",
    "a professional photograph of", "an artistic rendering of",
    "a beautiful photo of", "a studio portrait of",
]

adjectives = [
    "", "a majestic", "a young", "a beautiful", "a curious",
    "a large", "a wild", "a small", "a graceful", "a friendly",
    "a playful", "a sleepy", "a proud", "a gentle", "a fierce",
]

actions = [
    "standing", "running", "walking", "resting", "eating",
    "looking at the camera", "grazing", "playing",
    "drinking water", "standing tall", "sitting",
    "lying down", "jumping", "exploring",
]

scenes = [
    "in a green field", "in the African savanna", "in a forest",
    "on a mountain", "by a river", "in a zoo", "on a beach",
    "in the rain", "at sunset", "in a meadow", "under a tree",
    "in the snow", "on a rocky hill", "in a garden",
    "near a waterfall", "in a desert", "on a dirt road",
    "in tall grass", "at a watering hole", "in the mist",
    "at sunrise", "in a valley", "on a hillside",
    "near a pond", "in a national park", "in the wilderness",
    "on the plains", "in a clearing", "by a stream",
    "at a safari", "in the countryside",
]

contexts = [
    "", "during golden hour", "on a foggy morning",
    "with dramatic lighting", "in soft light",
    "with mountains in the background",
    "with a clear blue sky", "under stormy clouds",
]

In [ ]:
# ============================================================
# GENERATE PROMPT PAIRS
# ============================================================

def make_prompt(animal, style, adj, action, scene, ctx):
    """Assemble a prompt from parts."""
    if adj:
        core = f"{style} {adj} {animal} {action} {scene}"
    else:
        core = f"{style} a {animal} {action} {scene}"
    if ctx:
        core = f"{core}, {ctx}"
    return core


def generate_prompt_pairs(n_per_animal=PROMPTS_PER_ANIMAL):
    pairs = []       # (original_prompt, target_prompt, source_animal, target_animal)
    for animal in all_animals:
        target_animal = swap_map.get(animal, animal)
        for _ in range(n_per_animal):
            s = random.choice(styles)
            a = random.choice(adjectives)
            act = random.choice(actions)
            sc = random.choice(scenes)
            c = random.choice(contexts)

            orig = make_prompt(animal, s, a, act, sc, c)
            tgt  = make_prompt(target_animal, s, a, act, sc, c)
            pairs.append((orig, tgt, animal, target_animal))

    random.shuffle(pairs)
    return pairs


prompt_pairs = generate_prompt_pairs()
print(f"Total prompt pairs: {len(prompt_pairs):,}")
print(f"  Swap pairs:    {sum(1 for _,_,a,t in prompt_pairs if a != t):,}")
print(f"  Identity pairs: {sum(1 for _,_,a,t in prompt_pairs if a == t):,}")
print(f"\nExamples:")
for orig, tgt, src, dst in prompt_pairs[:3]:
    tag = "SWAP" if src != dst else "SAME"
    print(f"  [{tag}] {orig[:65]}")
    print(f"       → {tgt[:65]}\n")

---
## Part 2: Encode All Prompts

In [ ]:
# ============================================================
# ENCODE EVERY PROMPT TO (77, 768) EMBEDDINGS
# ============================================================

@torch.no_grad()
def encode_prompts(prompts, batch_size=128):
    all_emb = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        tok = tokenizer(batch, padding="max_length",
                        max_length=tokenizer.model_max_length,
                        truncation=True, return_tensors="pt")
        emb = text_encoder(tok.input_ids.to(device))[0]
        all_emb.append(emb.cpu())
    return torch.cat(all_emb, dim=0)


print("Encoding original prompts...")
orig_embs = encode_prompts([p[0] for p in prompt_pairs])
print("Encoding target prompts...")
tgt_embs  = encode_prompts([p[1] for p in prompt_pairs])

# Target animal class indices (for phase 2)
tgt_classes = torch.tensor([animal2idx[p[3]] for p in prompt_pairs])

print(f"Embeddings: {orig_embs.shape}  |  Classes: {tgt_classes.shape}")

---
## Part 3: Transformer Embedding Mapper

Why a Transformer instead of a CNN? The text embedding is a **sequence** of 77 token vectors. Self-attention lets every token attend to every other — so the model can learn that when it sees a "giraffe" token, it should also adjust the surrounding context tokens ("savanna", "tall", "spots") to be consistent with "zebra" ("stripes", "herd").

A CNN with kernel=3 only sees 3 neighboring tokens at a time — it misses these long-range dependencies.

In [ ]:
# ============================================================
# TRANSFORMER EMBEDDING MAPPER
# ============================================================

import warnings

class TransformerMapper(nn.Module):
    """BERT-style Transformer encoder that remaps (77, 768) text embeddings.

    Self-attention lets each token attend to the full sequence, so the model
    can learn global concept swaps (giraffe→zebra) while preserving structure.

    Residual design: output = input + scale * transformer(input)
    Starts as near-identity (scale ≈ 0.01) so untouched animals pass through.
    """
    def __init__(self, d_model=MAPPER_DIM, nhead=MAPPER_HEADS,
                 num_layers=MAPPER_LAYERS, dim_feedforward=MAPPER_FFN,
                 dropout=MAPPER_DROPOUT):
        super().__init__()

        self.input_norm = nn.LayerNorm(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            self.transformer = nn.TransformerEncoder(
                encoder_layer, num_layers=num_layers,
                enable_nested_tensor=False,
            )

        self.output_proj = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
        )

        self.scale = nn.Parameter(torch.tensor(0.01))

    def forward(self, x):
        """x: (B, 77, 768) → (B, 77, 768)"""
        h = self.input_norm(x)
        h = self.transformer(h)
        h = self.output_proj(h)
        return x + self.scale * h


mapper = TransformerMapper().to(device)
n_params = sum(p.numel() for p in mapper.parameters())
print(f"Transformer Mapper: {n_params:,} parameters ({n_params/1e6:.1f}M)")
print(f"  Layers: {MAPPER_LAYERS}, Heads: {MAPPER_HEADS}, FFN: {MAPPER_FFN}")

with torch.no_grad():
    test_in = orig_embs[:2].to(device)
    test_out = mapper(test_in)
    print(f"  Init residual magnitude: {(test_out - test_in).abs().mean():.6f}")

---
## Part 4: Phase 1 — Embedding Alignment (MSE)

Warm up the mapper with pure embedding supervision. Fast, gets the mapper 80% of the way there.

In [ ]:
# ============================================================
# PHASE 1: TRAIN ON EMBEDDING MSE
# ============================================================

train_ds = TensorDataset(orig_embs, tgt_embs)
train_dl = DataLoader(train_ds, batch_size=P1_BATCH, shuffle=True, drop_last=True)

optimizer = optim.AdamW(mapper.parameters(), lr=P1_LR, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=P1_EPOCHS)
p1_losses = []

mapper.train()
pbar = tqdm(range(P1_EPOCHS), desc="Phase 1")
for epoch in pbar:
    epoch_loss = 0
    n_batches = 0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        pred = mapper(x)
        loss = F.mse_loss(pred, y)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(mapper.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1
        p1_losses.append(loss.item())
    sched.step()
    pbar.set_postfix(loss=f"{epoch_loss/n_batches:.6f}")

plt.figure(figsize=(8, 3))
plt.plot(p1_losses, color='steelblue', alpha=0.7)
plt.xlabel('Step'); plt.ylabel('MSE'); plt.title('Phase 1: Embedding Alignment Loss')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# QUICK CHECK: embedding similarity after Phase 1
# ============================================================

mapper.eval()
checks = [
    ("a photo of a giraffe in the savanna", "a photo of a zebra in the savanna"),
    ("a photo of a zebra in a field",       "a photo of a giraffe in a field"),
    ("a photo of a bear by a lake",         "a photo of a bear by a lake"),
]
for orig_t, tgt_t in checks:
    o = encode_prompts([orig_t]).to(device)
    t = encode_prompts([tgt_t]).to(device)
    with torch.no_grad():
        m = mapper(o)
    sim_before = F.cosine_similarity(o.flatten(1), t.flatten(1)).item()
    sim_after  = F.cosine_similarity(m.flatten(1), t.flatten(1)).item()
    print(f"  '{orig_t[:40]}...'  before={sim_before:.4f}  after={sim_after:.4f}")

---
## Part 5: Build the Discriminator

A pretrained CLIP can classify animals, but it has never seen miniSD's specific image style. We fix this by:

1. Generating 40 images per animal class with the base SD model
2. Fine-tuning a linear classifier head on top of frozen CLIP image features

This gives us a classifier that understands what each animal looks like **in SD-generated images specifically**.

In [ ]:
# ============================================================
# GENERATE SD IMAGES FOR DISCRIMINATOR TRAINING
# ============================================================

disc_images = []   # PIL images
disc_labels = []   # integer class indices

disc_prompts_per_class = [
    "a photo of a {animal}",
    "a {animal} in the wild",
    "a close-up of a {animal}",
    "a {animal} standing in a field",
    "a {animal} in its natural habitat",
    "a painting of a {animal}",
    "a {animal} at sunset",
    "a {animal} by a river",
]

print(f"Generating {DISC_IMAGES_PER_CLASS} SD images per class ({NUM_CLASSES} classes)...")
for animal in tqdm(all_animals, desc="Animals"):
    cls_idx = animal2idx[animal]
    for i in range(DISC_IMAGES_PER_CLASS):
        template = disc_prompts_per_class[i % len(disc_prompts_per_class)]
        prompt = template.format(animal=animal)
        img = pipe(
            prompt, num_inference_steps=20, guidance_scale=7.5,
            generator=torch.Generator(device).manual_seed(SEED + cls_idx * 1000 + i),
        ).images[0]
        disc_images.append(img)
        disc_labels.append(cls_idx)

disc_labels = torch.tensor(disc_labels)
print(f"Generated {len(disc_images)} discriminator training images")

# Show a grid
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for i, (ax, animal) in enumerate(zip(axes.flat, all_animals)):
    idx = disc_labels.tolist().index(i)
    ax.imshow(disc_images[idx])
    ax.set_title(animal, fontsize=9)
    ax.axis('off')
plt.suptitle('SD-generated training images for discriminator (1 per class)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# FINE-TUNE A CLIP-BASED CLASSIFIER ON SD IMAGES
# ============================================================

# Load CLIP for feature extraction
clip_disc, _, clip_disc_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k", device=device,
)
clip_disc.eval()
for p in clip_disc.parameters():
    p.requires_grad_(False)

# Extract frozen CLIP features for all discriminator images
print("Extracting CLIP features...")
disc_features = []
with torch.no_grad():
    for img in tqdm(disc_images, desc="CLIP encode"):
        img_t = clip_disc_preprocess(img).unsqueeze(0).to(device)
        feat = clip_disc.encode_image(img_t)
        feat = F.normalize(feat, dim=-1)
        disc_features.append(feat.cpu())

disc_features = torch.cat(disc_features, dim=0)     # (N, 512)
clip_feat_dim = disc_features.shape[1]
print(f"CLIP features: {disc_features.shape}")

# Train a linear classifier head on top
classifier_head = nn.Linear(clip_feat_dim, NUM_CLASSES).to(device)

cls_dataset = TensorDataset(disc_features, disc_labels)
cls_loader  = DataLoader(cls_dataset, batch_size=32, shuffle=True, drop_last=True)
cls_opt     = optim.Adam(classifier_head.parameters(), lr=DISC_LR)

classifier_head.train()
for epoch in range(DISC_EPOCHS):
    correct, total_loss, total = 0, 0.0, 0
    for feats, labels in cls_loader:
        feats, labels = feats.to(device), labels.to(device)
        logits = classifier_head(feats)
        loss = F.cross_entropy(logits, labels)

        cls_opt.zero_grad()
        loss.backward()
        cls_opt.step()

        correct += (logits.argmax(1) == labels).sum().item()
        total_loss += loss.item() * len(labels)
        total += len(labels)

    if (epoch + 1) % 5 == 0:
        print(f"  Disc epoch {epoch+1}/{DISC_EPOCHS}  loss: {total_loss/total:.4f}  acc: {correct/total:.1%}")

# Freeze the classifier
classifier_head.eval()
classifier_head.requires_grad_(False)
print(f"\nDiscriminator frozen. Final accuracy: {correct/total:.1%}")

---
## Part 6: Phase 2 — Discriminator-Guided Training

Now the key innovation. For each training step:

1. Encode a prompt, remap it with the mapper
2. **One-step denoising:** sample noise at a moderate timestep, run UNet once, predict the clean image (x₀)
3. Decode x₀ with the VAE
4. Pass through frozen CLIP → frozen classifier → classification loss
5. Combine: `loss = MSE_embedding + λ × cross_entropy_classifier`
6. Backprop through classifier → CLIP → VAE → UNet → mapper

The UNet and VAE are frozen but we backprop **through** them to train the mapper. The one-step trick avoids the 30-step sampling cost.

In [ ]:
# ============================================================
# HELPER: Differentiable one-step image prediction
# ============================================================

# Preprocessing for CLIP (differentiable version using grid_sample)
CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1, 3, 1, 1)
CLIP_STD  = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1)


def one_step_predict_image(remapped_emb, timestep):
    """Run one UNet denoising step and predict the clean image.

    Returns: image tensor (1, 3, H, W) in [0, 1], differentiable w.r.t. remapped_emb
    """
    # Sample random noise as the noisy latent
    noisy_latent = torch.randn((1, 4, 32, 32), device=device)

    # UNet predicts noise (frozen, but gradients flow through remapped_emb)
    t_tensor = torch.tensor([timestep], device=device)
    noise_pred = unet(noisy_latent, t_tensor,
                      encoder_hidden_states=remapped_emb).sample

    # Predict x0 from the noise prediction (DDPM formula)
    alpha_bar_t = alphas_cumprod[timestep]
    x0_pred = (noisy_latent - torch.sqrt(1 - alpha_bar_t) * noise_pred) / torch.sqrt(alpha_bar_t)

    # Decode latent to image with VAE
    image = vae.decode(x0_pred / vae.config.scaling_factor).sample
    image = (image / 2 + 0.5).clamp(0, 1)            # [-1,1] → [0,1]

    return image


def classify_image(image):
    """Run image through CLIP + classifier head. Differentiable.

    image: (1, 3, H, W) in [0, 1]
    Returns: logits (1, NUM_CLASSES)
    """
    # Resize to CLIP input size (224x224)
    img_resized = F.interpolate(image, size=(224, 224), mode='bilinear', align_corners=False)
    # Normalize with CLIP stats
    img_norm = (img_resized - CLIP_MEAN) / CLIP_STD
    # CLIP encode (frozen but differentiable)
    features = clip_disc.encode_image(img_norm)
    features = F.normalize(features, dim=-1)
    # Classify
    logits = classifier_head(features)
    return logits


print("Helpers ready.")

In [ ]:
# ============================================================
# PHASE 2: TRAIN MAPPER WITH EMBEDDING + CLASSIFIER LOSS
# ============================================================

# Use a subset of swap-only pairs for Phase 2 (classifier signal matters most for swaps)
p2_indices = [i for i, (_, _, src, tgt) in enumerate(prompt_pairs) if src != tgt]
# Also include some identity pairs to maintain control
id_indices = [i for i, (_, _, src, tgt) in enumerate(prompt_pairs) if src == tgt]
p2_indices += random.sample(id_indices, min(len(id_indices), len(p2_indices) // 2))
random.shuffle(p2_indices)

print(f"Phase 2 training pool: {len(p2_indices)} pairs")

optimizer2 = optim.AdamW(mapper.parameters(), lr=P2_LR, weight_decay=1e-4)
sched2 = optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=P2_STEPS)
scaler = GradScaler()

p2_losses_emb = []
p2_losses_cls = []
p2_accs = []

mapper.train()
for step in tqdm(range(P2_STEPS), desc="Phase 2"):
    idx = p2_indices[step % len(p2_indices)]

    x_orig = orig_embs[idx:idx+1].to(device)         # (1, 77, 768)
    y_emb  = tgt_embs[idx:idx+1].to(device)          # (1, 77, 768)
    y_cls  = tgt_classes[idx:idx+1].to(device)        # (1,) target animal class

    # Sample a random timestep in the moderate range
    t = random.randint(P2_TIMESTEP_LO, P2_TIMESTEP_HI)

    with autocast():
        # Forward through mapper
        remapped = mapper(x_orig)                     # (1, 77, 768)

        # Loss 1: embedding MSE
        loss_emb = F.mse_loss(remapped, y_emb)

        # Loss 2: classifier feedback via one-step denoising
        pred_image = one_step_predict_image(remapped, t)  # (1, 3, H, W)
        logits = classify_image(pred_image)            # (1, NUM_CLASSES)
        loss_cls = F.cross_entropy(logits, y_cls)

        loss = loss_emb + P2_LAMBDA_CLS * loss_cls

    optimizer2.zero_grad()
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer2)
    nn.utils.clip_grad_norm_(mapper.parameters(), 1.0)
    scaler.step(optimizer2)
    scaler.update()
    sched2.step()

    p2_losses_emb.append(loss_emb.item())
    p2_losses_cls.append(loss_cls.item())
    p2_accs.append((logits.argmax(1) == y_cls).float().mean().item())

    if (step + 1) % 200 == 0:
        emb_avg = np.mean(p2_losses_emb[-200:])
        cls_avg = np.mean(p2_losses_cls[-200:])
        acc_avg = np.mean(p2_accs[-200:])
        print(f"  Step {step+1}/{P2_STEPS}  emb: {emb_avg:.5f}  cls: {cls_avg:.3f}  acc: {acc_avg:.1%}")

# Plot losses
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
axes[0].plot(p2_losses_emb, alpha=0.5, color='steelblue')
axes[0].set_title('Embedding MSE'); axes[0].set_xlabel('Step')
axes[1].plot(p2_losses_cls, alpha=0.5, color='coral')
axes[1].set_title('Classifier CE'); axes[1].set_xlabel('Step')
axes[2].plot(pd.Series(p2_accs).rolling(50).mean(), color='green')
axes[2].set_title('Classifier Accuracy (rolling)'); axes[2].set_xlabel('Step')
for ax in axes: ax.grid(True, alpha=0.3)
plt.suptitle('Phase 2: Discriminator-Guided Training', fontsize=11)
plt.tight_layout(); plt.show()

---
## Part 7: Generate Images + Visual Check

In [ ]:
# ============================================================
# IMAGE GENERATION WITH MAPPER (full diffusion loop)
# ============================================================

@torch.no_grad()
def generate_with_mapper(prompt, mapper_model, seed=SEED):
    """Full diffusion generation with remapped embeddings."""
    # Encode + remap
    tok = tokenizer(prompt, padding="max_length",
                    max_length=tokenizer.model_max_length,
                    truncation=True, return_tensors="pt")
    text_emb = text_encoder(tok.input_ids.to(device))[0]
    text_emb = mapper_model(text_emb)

    # Unconditional embedding (no remap)
    uncond_tok = tokenizer("", padding="max_length",
                           max_length=tokenizer.model_max_length,
                           truncation=True, return_tensors="pt")
    uncond_emb = text_encoder(uncond_tok.input_ids.to(device))[0]
    combined_emb = torch.cat([uncond_emb, text_emb])

    # Denoise
    gen_scheduler = DDPMScheduler.from_pretrained(BASE_MODEL, subfolder="scheduler")
    gen_scheduler.set_timesteps(NUM_INFERENCE_STEPS)
    gen = torch.Generator(device).manual_seed(seed)
    latents = torch.randn((1, 4, 32, 32), generator=gen, device=device)
    latents = latents * gen_scheduler.init_noise_sigma

    for t in gen_scheduler.timesteps:
        lat_in = torch.cat([latents] * 2)
        lat_in = gen_scheduler.scale_model_input(lat_in, t)
        noise_pred = unet(lat_in, t, encoder_hidden_states=combined_emb).sample
        nu, nc = noise_pred.chunk(2)
        noise_pred = nu + GUIDANCE_SCALE * (nc - nu)
        latents = gen_scheduler.step(noise_pred, t, latents).prev_sample

    image = vae.decode(latents / vae.config.scaling_factor).sample
    image = (image / 2 + 0.5).clamp(0, 1)
    return T.ToPILImage()(image[0].float().cpu())


print("Generator ready.")

In [ ]:
# ============================================================
# VISUAL CHECK: before vs after
# ============================================================

mapper.eval()
check_prompts = [
    "a photo of a giraffe in the African savanna",
    "a photo of a zebra in a green field",
    "a photo of a bear by a mountain lake",
    "a painting of a giraffe at sunset",
]

fig, axes = plt.subplots(len(check_prompts), 2, figsize=(8, 4 * len(check_prompts)))
for i, p in enumerate(check_prompts):
    orig_img = pipe(p, num_inference_steps=NUM_INFERENCE_STEPS,
                    guidance_scale=GUIDANCE_SCALE,
                    generator=torch.Generator(device).manual_seed(SEED)).images[0]
    mapped_img = generate_with_mapper(p, mapper, seed=SEED)

    axes[i, 0].imshow(orig_img); axes[i, 0].set_title(f'Original', fontsize=9); axes[i, 0].axis('off')
    axes[i, 1].imshow(mapped_img); axes[i, 1].set_title(f'Mapper', fontsize=9); axes[i, 1].axis('off')
    axes[i, 0].set_ylabel(p[:35] + '...', fontsize=8, rotation=0, labelpad=120, va='center')

plt.suptitle('Left: original  |  Right: with Transformer mapper', fontsize=11)
plt.tight_layout(); plt.show()

---
## Part 8: Generate All Test Images + Submit

In [ ]:
# ============================================================
# GENERATE ALL TEST PROMPT IMAGES
# ============================================================

mapper.eval()
generated_images = []
for _, row in tqdm(test_prompts.iterrows(), total=len(test_prompts), desc="Generating"):
    img = generate_with_mapper(row["prompt"], mapper, seed=SEED)
    generated_images.append(img)

print(f"Generated {len(generated_images)} images")

In [ ]:
# ================================================================
# CLIP EVALUATION — DO NOT MODIFY THIS CELL
# ================================================================

clip_eval, _, clip_eval_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k", device=device,
)
clip_eval_tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_eval.eval()

similarities = []
for idx, row in tqdm(test_prompts.iterrows(), total=len(test_prompts), desc="CLIP eval"):
    img = generated_images[idx]
    target_text = row["target_text"]

    img_tensor = clip_eval_preprocess(img).unsqueeze(0).to(device)
    text_tokens = clip_eval_tokenizer([target_text]).to(device)

    with torch.no_grad():
        img_features = clip_eval.encode_image(img_tensor)
        txt_features = clip_eval.encode_text(text_tokens)
        img_features = F.normalize(img_features, dim=-1)
        txt_features = F.normalize(txt_features, dim=-1)
        sim = (img_features @ txt_features.T).item()

    similarities.append(sim * 100)

similarities = np.array(similarities)
print(f"\nMean CLIP Similarity (score): {similarities.mean():.2f}")
print(f"Min: {similarities.min():.2f}  Max: {similarities.max():.2f}")
for prefix in ["giraffe", "zebra", "ctrl", "mixed"]:
    mask = test_prompts["id"].str.startswith(prefix)
    print(f"  {prefix:10s}: {similarities[mask.values].mean():.2f}")

In [ ]:
# ================================================================
# GENERATE SUBMISSION — DO NOT MODIFY THIS CELL
# ================================================================

def generate_submission(similarities, filename="submission.csv"):
    sims = np.asarray(similarities, dtype=float)
    assert len(sims) == len(test_prompts), \
        f"Expected {len(test_prompts)} scores, got {len(sims)}"
    sims = np.clip(sims, 0.0, 100.0)
    submission = pd.DataFrame({
        "id": test_prompts["id"].values,
        "prediction": sims,
    })
    submission.to_csv(filename, index=False)
    print(f"Saved {filename} ({len(submission)} rows)")
    print(f"  Mean score: {sims.mean():.2f}")
    return submission

submission = generate_submission(similarities)

---
## Architecture Summary

| Component | Role | Trainable? |
|---|---|---|
| Text encoder (CLIP) | Prompt → embedding | Frozen |
| **Transformer Mapper** | Remap embedding (giraffe→zebra) | **Trained** |
| UNet | Denoise latent (conditioned on remapped embedding) | Frozen |
| VAE | Decode latent → image | Frozen |
| CLIP (discriminator backbone) | Image → features | Frozen |
| Classifier head | Features → animal class | Frozen (fine-tuned in Part 5) |

**Phase 1 loss:** MSE between mapper output and target embedding (fast, supervised)

**Phase 2 loss:** MSE + λ × CrossEntropy from classifier on one-step denoised images (adds image-level signal)

**Key ideas:**
- Self-attention in the mapper captures long-range token dependencies that CNN misses
- The discriminator is fine-tuned on SD-generated images so it understands the UNet's visual style
- One-step denoising makes end-to-end training feasible on T4 (no 30-step backprop)
- Residual design with learned scale lets the mapper start as identity and learn only the delta